In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

SEED = 42
RNG = np.random.default_rng(SEED)
SIM_DAYS = 60

# ---------------- PARAMETERS ----------------

sku_list = ["Potato","Ivy_gourd","Cucumber","Pumpkin","Bitter_gourd","Brinjal","Beans","Capsicum","Madras_Cucumber","Carrot","Sweet_Potato",
"Ridge_gourd","Plantain","Beetroot","Cabbage","Curry_leaves", "Coriander", "Pudina","Tomato","Green_Chillies","Onion"]

lead_time = {sku:1 for sku in sku_list}

shelf_life = {"Potato":7,"Ivy_gourd":3,"Cucumber":4,"Pumpkin":10,"Bitter_gourd":3,"Brinjal":2,"Beans":2,"Capsicum":4,"Madras_Cucumber":4,
"Carrot":5,"Sweet_Potato":7,"Ridge_gourd":2,"Plantain":6,"Beetroot":7,"Cabbage":5,"Curry_leaves":2, "Coriander":1, "Pudina":3,"Tomato":3,"Green_Chillies":3,"Onion":30}

order_interval = {sku:1 for sku in sku_list}

# ---------------- UPLOAD FILE ----------------

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ---------------- READ DATA ----------------

demand_df = pd.read_excel(filename)
profit_df = pd.read_excel(filename,sheet_name="ProfitPerKg")
lostsales_df = pd.read_excel(filename,sheet_name="LostSales")
expired_df = pd.read_excel(filename,sheet_name="Expired")
unitcost_df = pd.read_excel(filename,sheet_name="UnitCost")

train_df = demand_df[sku_list]

# ---------------- DISTRIBUTION FITTING ----------------

best_fit={}

print("\n===== DISTRIBUTION FITTING =====")

for sku in sku_list:

    data=train_df[sku].values

    distributions=[stats.norm,stats.expon,stats.gamma,stats.triang,stats.beta,stats.lognorm,stats.uniform]

    best_p=-1

    for dist in distributions:

        try:

            params=dist.fit(data)

            ks_stat,p=stats.kstest(data,dist.name,args=params)

            if p>best_p:

                best_p=p
                best_fit[sku]=(dist.name,params,ks_stat,p)

        except:
            continue

    name,params,ks,p=best_fit[sku]

    print("\nVegetable:",sku)
    print("Best Distribution:",name)

    if name=="norm":
        mu,sigma=params
        print("Mean (μ):",mu)
        print("Variance (σ²):",sigma**2)

    elif name=="uniform":
        a,b=params
        print("Lower bound (a):",a)
        print("Range (b):",b)

    elif name=="gamma":
        k,loc,theta=params
        print("Shape (k):",k)
        print("Scale (θ):",theta)

    elif name=="lognorm":
        sigma,loc,scale=params
        print("Sigma:",sigma)

    elif name=="beta":
        a,b,loc,scale=params
        print("Alpha:",a)
        print("Beta:",b)

    elif name=="triang":
        c,loc,scale=params
        print("Shape (c):",c)

    print("KS Statistic:",round(ks,4))
    print("KS p-value:",round(p,4))

    if p>0.05:
        print("Fit Status: GOOD FIT")
    else:
        print("Fit Status: POOR FIT")

# ---------------- DEMAND SIMULATION ----------------

demand_sim=[]

for d in range(SIM_DAYS):

    row={}

    for sku in sku_list:

        name,params,_,_=best_fit[sku]

        dist=getattr(stats,name)

        val=dist.rvs(*params,random_state=RNG)

        row[sku]=max(0,int(val))

    demand_sim.append(row)

# ---------------- INVENTORY SIMULATION ----------------

class InventorySim:

    def __init__(self):

        self.reset()
        self.target={sku:0 for sku in sku_list}

    def reset(self):

        self.inventory={sku:[] for sku in sku_list}
        self.pipeline={sku:[] for sku in sku_list}
        self.metrics=[]

    def place_order(self,sku,qty,day):

        arrival=day+lead_time[sku]
        self.pipeline[sku].append((arrival,qty))

    def step(self,day,demand):

        row={"day":day}
        total_profit=0

        # arrivals
        for sku in sku_list:

            arrivals=[q for a,q in self.pipeline[sku] if a==day]

            self.pipeline[sku]=[(a,q) for a,q in self.pipeline[sku] if a!=day]

            for q in arrivals:
                self.inventory[sku].append([q,day])

        # demand
        for sku in sku_list:

            need=demand[sku]
            served=0

            self.inventory[sku].sort(key=lambda x:x[1], reverse=True)

            while need>0 and self.inventory[sku]:

                qty,arr=self.inventory[sku][0]

                take=min(qty,need)

                served+=take
                need-=take

                if take<qty:
                    self.inventory[sku][0][0]-=take
                else:
                    self.inventory[sku].pop(0)

            lost=need

            row[f"{sku}_served"]=served
            row[f"{sku}_lost_sales"]=lost

        # expiry
        for sku in sku_list:

            expired=0
            remain=[]

            for qty,arr in self.inventory[sku]:

                expiry=arr+shelf_life[sku]

                if day>expiry:
                    expired+=qty
                else:
                    remain.append([qty,arr])

            self.inventory[sku]=remain

            row[f"{sku}_expired"]=expired

            served=row[f"{sku}_served"]

            day_idx = (day-1) % len(train_df)
            profit=served*profit_df.loc[day_idx,sku]-expired*unitcost_df.loc[day_idx,sku]

            row[f"{sku}_profit"]=profit

            total_profit+=profit

        row["total_profit"]=total_profit

        # ordering
        for sku in sku_list:

            on_hand=sum(q for q,_ in self.inventory[sku])
            pipeline=sum(q for _,q in self.pipeline[sku])

            pos=on_hand+pipeline

            if self.policy_type[sku]=="target":
                 order=max(0,self.target[sku]-pos)

            else:
                 order=self.fixed_qty[sku]

            if order>0:
                self.place_order(sku,order,day)

        self.metrics.append(row)

    def run(self,demand):

        self.reset()

        for sku in sku_list:
            self.place_order(sku,self.target[sku],0)

        for d in range(1,len(demand)+1):
            self.step(d,demand[d-1])

        return pd.DataFrame(self.metrics)

# -----------------------------------#
# convert raw excel file in df_real with day-wise operational performance

df_real=pd.DataFrame()

for day in range(len(train_df)):

    row={"day":day+1}
    total=0

    for sku in sku_list:

        demand=train_df.loc[day,sku]
        lost=lostsales_df.loc[day,sku]
        expired=expired_df.loc[day,sku]

        served=demand-lost

        profit=served*profit_df.loc[day,sku]-expired*unitcost_df.loc[day,sku]

        row[f"{sku}_lost_sales"]=lost
        row[f"{sku}_expired"]=expired
        row[f"{sku}_profit"]=profit

        total+=profit

    row["total_profit"]=total

    df_real=pd.concat([df_real,pd.DataFrame([row])])

# --------------- CONFIDENCE INTERVAL VALIDATION----------------#

validation_rows = []

def CI(data):

    m=np.mean(data)
    s=np.std(data,ddof=1)
    n=len(data)

    margin=2.576*s/np.sqrt(n)

    return m-margin,m+margin

# average after removing warm-up days based on shelf life
def post_warmup_avg(df, sku, metric):
    start_day = shelf_life[sku] + 1
    col = f"{sku}_{metric}"
    return df[df["day"] >= start_day][col].mean()

# total after removing warm-up days based on shelf life
def post_warmup_sum(df, sku, metric):
    start_day = shelf_life[sku] + 1
    col = f"{sku}_{metric}"
    return df[df["day"] >= start_day][col].sum()

# ---------------- POLICY SEARCH ----------------

sim=InventorySim()

policy={}
policy_name={}
selected_policy_results = {}   # >>> ADD THIS


# >>> NEW: save current-policy validation outputs once, reuse later
current_policy_validation_saved = {}

print("\n===== ORDERING POLICY SELECTED =====")

for sku in sku_list:

    mean = train_df[sku].mean()

    candidates = {
        "0.50 Mean": int(max(0, 0.50 * mean)),
        "0.75 Mean": int(max(0, 0.75 * mean)),
        "0.90 Mean": int(max(0, 0.90 * mean)),
        "Mean": int(max(0, mean)),
        "1.10 Mean": int(max(0, 1.10 * mean)),
        "1.25 Mean": int(max(0, 1.25 * mean)),
        "1.5 Mean": int(max(0, 1.5 * mean)),
        "1.75 Mean": int(max(0, 1.75 * mean)),
        "P60": int(train_df[sku].quantile(0.60)),
        "P75": int(train_df[sku].quantile(0.75)),
        "P85": int(train_df[sku].quantile(0.85)),
        "P90": int(train_df[sku].quantile(0.90)),
        "P95": int(train_df[sku].quantile(0.95))
    }

    best_score = -1
    best_policy = None
    best_value = None
    best_type = None

    print("\n=================================================")
    print(f"VEGETABLE: {sku}")
    print("=================================================")

    for name, val in candidates.items():

        for ptype in ["target", "fixed"]:

            profit_runs = []
            lost_runs = []
            exp_runs = []

            for r in range(10):

                run_rng = np.random.default_rng(SEED + r)
                demand_sim_new = []

                for d in range(SIM_DAYS):
                    row = {}

                    for s in sku_list:
                        dist_name, params, _, _ = best_fit[s]
                        dist = getattr(stats, dist_name)

                        try:
                            val_sim = dist.rvs(*params, random_state=run_rng)
                        except:
                            val_sim = train_df[s].mean()

                        row[s] = max(0, int(round(val_sim)))

                    demand_sim_new.append(row)

                sim.target = {s: int(train_df[s].mean()) for s in sku_list}
                sim.policy_type = {s: "target" for s in sku_list}
                sim.fixed_qty = {s: int(train_df[s].mean()) for s in sku_list}

                if ptype == "target":
                    sim.target[sku] = val
                    sim.policy_type[sku] = "target"
                else:
                    sim.fixed_qty[sku] = val
                    sim.policy_type[sku] = "fixed"

                df = sim.run(demand_sim_new)

                profit_runs.append(post_warmup_avg(df, sku, "profit"))
                lost_runs.append(post_warmup_avg(df, sku, "lost_sales"))
                exp_runs.append(post_warmup_avg(df, sku, "expired"))

            sim_profit = np.mean(profit_runs)
            sim_lost = np.mean(lost_runs)
            sim_exp = np.mean(exp_runs)

            p_low, p_high = CI(df_real[f"{sku}_profit"])
            l_low, l_high = CI(df_real[f"{sku}_lost_sales"])
            e_low, e_high = CI(df_real[f"{sku}_expired"])

            profit_ok = p_low <= sim_profit <= p_high
            lost_ok = l_low <= sim_lost <= l_high
            exp_ok = e_low <= sim_exp <= e_high

            score = int(profit_ok) + int(lost_ok) + int(exp_ok)

            if score > best_score:
                best_score = score
                best_policy = name
                best_value = val
                best_type = ptype

                best_sim_profit = sim_profit
                best_sim_lost = sim_lost
                best_sim_exp = sim_exp

                best_p_low, best_p_high = p_low, p_high
                best_l_low, best_l_high = l_low, l_high
                best_e_low, best_e_high = e_low, e_high

                best_profit_ok = profit_ok
                best_lost_ok = lost_ok
                best_exp_ok = exp_ok

            elif score == best_score:

                if profit_ok and not best_profit_ok:
                    best_policy = name
                    best_value = val
                    best_type = ptype

                    best_sim_profit = sim_profit
                    best_sim_lost = sim_lost
                    best_sim_exp = sim_exp

                    best_p_low, best_p_high = p_low, p_high
                    best_l_low, best_l_high = l_low, l_high
                    best_e_low, best_e_high = e_low, e_high

                    best_profit_ok = profit_ok
                    best_lost_ok = lost_ok
                    best_exp_ok = exp_ok

                elif profit_ok == best_profit_ok and sim_profit > best_sim_profit:
                    best_policy = name
                    best_value = val
                    best_type = ptype

                    best_sim_profit = sim_profit
                    best_sim_lost = sim_lost
                    best_sim_exp = sim_exp

                    best_p_low, best_p_high = p_low, p_high
                    best_l_low, best_l_high = l_low, l_high
                    best_e_low, best_e_high = e_low, e_high

                    best_profit_ok = profit_ok
                    best_lost_ok = lost_ok
                    best_exp_ok = exp_ok

    policy[sku] = best_value
    policy_name[sku] = (best_policy, best_type, best_score)

    # >>> FIX ADDED HERE
    selected_policy_results[sku] = {
        "Sim Profit": best_sim_profit,
        "Sim Lost Sales": best_sim_lost,
        "Sim Expiry": best_sim_exp
    }

    # >>> NEW: SAVE CURRENT POLICY VALIDATION OUTPUTS FOR REUSE
    current_policy_validation_saved[sku] = {
        "Profit CI Low": best_p_low,
        "Profit CI High": best_p_high,
        "Sim Profit Mean": best_sim_profit,
        "Profit Valid": best_profit_ok,

        "Lost CI Low": best_l_low,
        "Lost CI High": best_l_high,
        "Sim Lost Mean": best_sim_lost,
        "Lost Valid": best_lost_ok,

        "Expiry CI Low": best_e_low,
        "Expiry CI High": best_e_high,
        "Sim Expiry Mean": best_sim_exp,
        "Expiry Valid": best_exp_ok,

        "Validation Score": best_score
    }

    print("\n***** SELECTED POLICY *****")

    print(f"{sku} → {best_type.upper()} policy")
    print(f"Candidate: {best_policy}")
    print(f"Order Quantity: {best_value}")

    print("\nPROFIT")
    print(f"99% CI = [{round(best_p_low,2)}, {round(best_p_high,2)}]")
    print(f"Simulated Value (15-run avg) = {round(best_sim_profit,2)}")
    print(f"Within CI: {best_profit_ok}")

    print("\nLOST SALES")
    print(f"99% CI = [{round(best_l_low,2)}, {round(best_l_high,2)}]")
    print(f"Simulated Value (15-run avg) = {round(best_sim_lost,2)}")
    print(f"Within CI: {best_lost_ok}")

    print("\nEXPIRY")
    print(f"99% CI = [{round(best_e_low,2)}, {round(best_e_high,2)}]")
    print(f"Simulated Value (15-run avg) = {round(best_sim_exp,2)}")
    print(f"Within CI: {best_exp_ok}")

    print(f"\nCI Matches = {best_score}/3")

# ---------------- FINAL SIMULATION WITH SELECTED POLICIES ----------------

sim.target = {sku: policy[sku] for sku in sku_list}
sim.policy_type = {sku: policy_name[sku][1] for sku in sku_list}
sim.fixed_qty = {sku: policy[sku] for sku in sku_list}

df_sim = sim.run(demand_sim)

# ---------------- REAL DATA TABLE ----------------

print("\n===== REAL DATA (10 DAYS) =====")
print(df_real.to_string(index=False))

print("\n===== SIMULATED DATA (60 DAYS) =====")
print(df_sim.to_string(index=False))

# ---------------- AVERAGE COMPARISON ----------------
# USE SAME 15-RUN AVERAGE RESULTS AS USED IN POLICY VALIDATION

summary=[]

for sku in sku_list:

    summary.append({

        "Vegetable": sku,

        "Real Avg Lost": df_real[f"{sku}_lost_sales"].mean(),
        "Sim Avg Lost": current_policy_validation_saved[sku]["Sim Lost Mean"],

        "Real Avg Expiry": df_real[f"{sku}_expired"].mean(),
        "Sim Avg Expiry": current_policy_validation_saved[sku]["Sim Expiry Mean"],

        "Real Avg Profit": df_real[f"{sku}_profit"].mean(),
        "Sim Avg Profit": current_policy_validation_saved[sku]["Sim Profit Mean"]
    })

summary_df = pd.DataFrame(summary)

print("\n===== AVERAGE COMPARISON =====")
print(summary_df.to_string(index=False))

# ============================================================
# DIRECT 15-RUN VALIDATION USING SAVED RESULTS
# ============================================================

print("\n===== 99% CONFIDENCE INTERVAL VALIDATION (DIRECT 10-RUN AVERAGE) =====")

validation_rows = []

for sku in sku_list:

    p_low = current_policy_validation_saved[sku]["Profit CI Low"]
    p_high = current_policy_validation_saved[sku]["Profit CI High"]
    sim_profit = current_policy_validation_saved[sku]["Sim Profit Mean"]
    profit_valid = current_policy_validation_saved[sku]["Profit Valid"]

    l_low = current_policy_validation_saved[sku]["Lost CI Low"]
    l_high = current_policy_validation_saved[sku]["Lost CI High"]
    sim_lost = current_policy_validation_saved[sku]["Sim Lost Mean"]
    lost_valid = current_policy_validation_saved[sku]["Lost Valid"]

    e_low = current_policy_validation_saved[sku]["Expiry CI Low"]
    e_high = current_policy_validation_saved[sku]["Expiry CI High"]
    sim_exp = current_policy_validation_saved[sku]["Sim Expiry Mean"]
    exp_valid = current_policy_validation_saved[sku]["Expiry Valid"]

    validation_score = current_policy_validation_saved[sku]["Validation Score"]

    if validation_score == 3:
        validation_type = "Fully Validated"
    elif validation_score == 2:
        validation_type = "Partially Validated (2/3)"
    elif validation_score == 1:
        validation_type = "Weakly Validated (1/3)"
    else:
        validation_type = "Not Validated"

    profit_pct = ((sim_profit - df_real[f"{sku}_profit"].mean()) / df_real[f"{sku}_profit"].mean() * 100) if df_real[f"{sku}_profit"].mean() != 0 else np.nan
    lost_pct = ((sim_lost - df_real[f"{sku}_lost_sales"].mean()) / df_real[f"{sku}_lost_sales"].mean() * 100) if df_real[f"{sku}_lost_sales"].mean() != 0 else np.nan
    exp_pct = ((sim_exp - df_real[f"{sku}_expired"].mean()) / df_real[f"{sku}_expired"].mean() * 100) if df_real[f"{sku}_expired"].mean() != 0 else np.nan

    validation_rows.append({
        "Vegetable": sku,

        "Profit CI (99%)": f"[{round(p_low,2)}, {round(p_high,2)}]",
        "Sim Profit Mean (15 runs)": round(sim_profit,2),
        "Profit % Diff": round(profit_pct,2) if not np.isnan(profit_pct) else "NA",
        "Profit Valid": profit_valid,

        "Lost Sales CI (99%)": f"[{round(l_low,2)}, {round(l_high,2)}]",
        "Sim Lost Mean (15 runs)": round(sim_lost,2),
        "Lost % Diff": round(lost_pct,2) if not np.isnan(lost_pct) else "NA",
        "Lost Valid": lost_valid,

        "Expiry CI (99%)": f"[{round(e_low,2)}, {round(e_high,2)}]",
        "Sim Expiry Mean (15 runs)": round(sim_exp,2),
        "Expiry % Diff": round(exp_pct,2) if not np.isnan(exp_pct) else "NA",
        "Expiry Valid": exp_valid,

        "Validation Score": validation_score,
        "Validation Type": validation_type
    })

validation_15_df = pd.DataFrame(validation_rows)

print(validation_15_df.to_string(index=False))

# ============================================================
# VALIDATION GROUPS
# ============================================================

print("\n===== FULLY VALIDATED VEGETABLES (3/3) =====")
print(validation_15_df[validation_15_df["Validation Score"] == 3].to_string(index=False))

print("\n===== PARTIALLY VALIDATED VEGETABLES (2/3) =====")
print(validation_15_df[validation_15_df["Validation Score"] == 2].to_string(index=False))

print("\n===== WEAKLY VALIDATED VEGETABLES (1/3) =====")
print(validation_15_df[validation_15_df["Validation Score"] == 1].to_string(index=False))

print("\n===== NOT VALIDATED VEGETABLES (0/3) =====")
print(validation_15_df[validation_15_df["Validation Score"] == 0].to_string(index=False))

# ============================================================
# KEEP ONLY FULLY VALIDATED VEGETABLES FOR FINAL OPTIMIZATION
# ============================================================

validated_skus = validation_15_df[validation_15_df["Validation Score"] == 3]["Vegetable"].tolist()

print("\n===== VEGETABLES USED FOR FINAL OPTIMIZATION =====")
print(validated_skus)
print("\n===== REMOVED (NOT VALIDATED) VEGETABLES =====")
print([sku for sku in sku_list if sku not in validated_skus])

# ============================================================
# ALSO SHOW 2/3 VALIDATED VEGETABLES
# ============================================================

validated_skus_2 = validation_15_df[validation_15_df["Validation Score"] == 2]["Vegetable"].tolist()

print("\n===== VEGETABLES VALIDATED IN 2/3 =====")
print(validated_skus_2)

# ============================================================
# FINAL POLICY OPTIMIZATION USING 10 ORDERING POLICIES
# Objective = MAXIMIZE AVERAGE PROFIT OVER 15 RUNS
# ============================================================

print("\n\n==============================================================")
print("===== FINAL BEST POLICY SEARCH (10 POLICIES, 10 RUNS) =====")
print("==============================================================")

best_policy_final = {}
best_policy_type_final = {}
best_policy_name_final = {}
best_policy_results_final = {}
best_policy_last_run_df = {}

for sku in sku_list:

    mean = train_df[sku].mean()

    policy_candidates = {
    "0.50 Mean": int(max(0, 0.50 * mean)),
    "0.75 Mean": int(max(0, 0.75 * mean)),
    "0.90 Mean": int(max(0, 0.90 * mean)),
    "Mean": int(max(0, mean)),
    "1.10 Mean": int(max(0, 1.10 * mean)),
    "1.25 Mean": int(max(0, 1.25 * mean)),
    "1.50 Mean": int(max(0, 1.50 * mean)),
    "1.75 Mean": int(max(0, 1.75 * mean)),
    "P60": int(train_df[sku].quantile(0.60)),
    "P75": int(train_df[sku].quantile(0.75)),
    "P85": int(train_df[sku].quantile(0.85)),
    "P90": int(train_df[sku].quantile(0.90)),
    "P95": int(train_df[sku].quantile(0.95)),
}

    best_avg_profit = -1e18
    best_avg_lost = None
    best_avg_exp = None
    best_name = None
    best_val = None
    best_type = None
    best_last_run = None

    print("\n--------------------------------------------------")
    print(f"VEGETABLE: {sku}")
    print("--------------------------------------------------")

    for pol_name, pol_val in policy_candidates.items():

        for ptype in ["target", "fixed"]:

            profit_runs = []
            lost_runs = []
            exp_runs = []

            last_run_df = None

            for r in range(10):

                demand_sim_new = []

                run_rng = np.random.default_rng(SEED + r)

                for d in range(SIM_DAYS):
                    row = {}

                    for s in sku_list:
                        name, params, _, _ = best_fit[s]
                        dist = getattr(stats, name)

                        try:
                            val = dist.rvs(*params, random_state=run_rng)
                        except:
                            val = train_df[s].mean()

                        row[s] = max(0, int(round(val)))

                    demand_sim_new.append(row)

                sim.reset()

                sim.target = {s: int(train_df[s].mean()) for s in sku_list}
                sim.policy_type = {s: "target" for s in sku_list}
                sim.fixed_qty = {s: int(train_df[s].mean()) for s in sku_list}

                if ptype == "target":
                    sim.target[sku] = pol_val
                    sim.policy_type[sku] = "target"
                else:
                    sim.fixed_qty[sku] = pol_val
                    sim.policy_type[sku] = "fixed"

                df_temp = sim.run(demand_sim_new)

                profit_runs.append(post_warmup_avg(df_temp, sku, "profit"))
                lost_runs.append(post_warmup_avg(df_temp, sku, "lost_sales"))
                exp_runs.append(post_warmup_avg(df_temp, sku, "expired"))

                if r == 9:
                    last_run_df = df_temp.copy()

            avg_profit = np.mean(profit_runs)
            avg_lost = np.mean(lost_runs)
            avg_exp = np.mean(exp_runs)

            print(f"{pol_name:<12} | {ptype.upper():<6} | Avg Profit = {avg_profit:.2f} | Avg Lost = {avg_lost:.2f} | Avg Expiry = {avg_exp:.2f}")

            if (
                (avg_profit > best_avg_profit) or
                (avg_profit == best_avg_profit and avg_lost < best_avg_lost) or
                (avg_profit == best_avg_profit and avg_lost == best_avg_lost and avg_exp < best_avg_exp)
            ):
                best_avg_profit = avg_profit
                best_avg_lost = avg_lost
                best_avg_exp = avg_exp
                best_name = pol_name
                best_val = pol_val
                best_type = ptype
                best_last_run = last_run_df.copy()

    best_policy_final[sku] = best_val
    best_policy_type_final[sku] = best_type
    best_policy_name_final[sku] = best_name
    best_policy_results_final[sku] = {
        "Avg Profit": best_avg_profit,
        "Avg Lost Sales": best_avg_lost,
        "Avg Expiry": best_avg_exp
    }
    best_policy_last_run_df[sku] = best_last_run

    print("\n***** FINAL SELECTED POLICY *****")
    print(f"{sku} → {best_type.upper()} policy")
    print(f"Policy Name: {best_name}")
    print(f"Order Quantity: {best_val}")
    print(f"Average Profit (15 runs): {best_avg_profit:.2f}")
    print(f"Average Lost Sales (15 runs): {best_avg_lost:.2f}")
    print(f"Average Expiry (15 runs): {best_avg_exp:.2f}")

# ============================================================
# SUMMARY TABLE OF FINAL BEST POLICIES
# ============================================================

final_policy_summary = []

for sku in validated_skus:
    final_policy_summary.append({
        "Vegetable": sku,
        "Selected Policy Type": best_policy_type_final[sku],
        "Selected Policy Name": best_policy_name_final[sku],
        "Order Quantity": best_policy_final[sku],
        "Avg Profit (15 runs)": round(best_policy_results_final[sku]["Avg Profit"], 2),
        "Avg Lost Sales (15 runs)": round(best_policy_results_final[sku]["Avg Lost Sales"], 2),
        "Avg Expiry (15 runs)": round(best_policy_results_final[sku]["Avg Expiry"], 2)
    })

final_policy_summary_df = pd.DataFrame(final_policy_summary)

print("\n\n==============================================================")
print("===== FINAL BEST POLICY SUMMARY =====")
print("==============================================================")
print(final_policy_summary_df.to_string(index=False))

# ============================================================
# SHOW HOW LAST RUN HAPPENED (DAY-WISE) FOR SELECTED POLICY
# ============================================================

print("\n\n==============================================================")
print("===== DAY-WISE LAST RUN OUTPUT FOR SELECTED BEST POLICIES =====")
print("==============================================================")

sim.reset()
sim.target = {s: best_policy_final[s] for s in sku_list}
sim.policy_type = {s: best_policy_type_final[s] for s in sku_list}
sim.fixed_qty = {s: best_policy_final[s] for s in sku_list}

run_rng = np.random.default_rng(SEED + 999)
demand_best_last_run = []

for d in range(SIM_DAYS):
    row = {}

    for s in sku_list:
        name, params, _, _ = best_fit[s]
        dist = getattr(stats, name)

        try:
            val = dist.rvs(*params, random_state=run_rng)
        except:
            val = train_df[s].mean()

        row[s] = max(0, int(round(val)))

    demand_best_last_run.append(row)

df_best_last_run = sim.run(demand_best_last_run)

print(df_best_last_run.to_string(index=False))

# ============================================================
# REAL DATA SIMULATION USING FINAL BEST ORDERING POLICY
# ============================================================

print("\n\n==============================================================")
print("===== REAL DATA PERFORMANCE USING FINAL BEST ORDERING POLICY =====")
print("==============================================================")

real_demand_input = []

for day in range(len(train_df)):
    row = {}
    for sku in sku_list:
        row[sku] = int(train_df.loc[day, sku])
    real_demand_input.append(row)

sim.reset()

sim.target = {s: best_policy_final[s] for s in sku_list}
sim.policy_type = {s: best_policy_type_final[s] for s in sku_list}
sim.fixed_qty = {s: best_policy_final[s] for s in sku_list}

df_best_real = sim.run(real_demand_input)

print("\n===== BEST ORDERING POLICY ON REAL DATA (DAY-WISE) =====")
print(df_best_real.to_string(index=False))

# ============================================================
# AVERAGE COMPARISON: CURRENT POLICY vs BEST POLICY (REAL DATA)
# ============================================================

comparison_rows = []

for sku in validated_skus:

    current_avg_lost = df_real[f"{sku}_lost_sales"].mean()
    best_avg_lost = df_best_real[f"{sku}_lost_sales"].mean()

    current_avg_exp = df_real[f"{sku}_expired"].mean()
    best_avg_exp = df_best_real[f"{sku}_expired"].mean()

    current_avg_profit = df_real[f"{sku}_profit"].mean()
    best_avg_profit = df_best_real[f"{sku}_profit"].mean()

    lost_diff = best_avg_lost - current_avg_lost
    exp_diff = best_avg_exp - current_avg_exp
    profit_diff = best_avg_profit - current_avg_profit

    lost_pct = ((lost_diff / current_avg_lost) * 100) if current_avg_lost != 0 else np.nan
    exp_pct = ((exp_diff / current_avg_exp) * 100) if current_avg_exp != 0 else np.nan
    profit_pct = ((profit_diff / current_avg_profit) * 100) if current_avg_profit != 0 else np.nan

    comparison_rows.append({
        "Vegetable": sku,

        "Current Avg Lost": round(current_avg_lost, 2),
        "Best Avg Lost": round(best_avg_lost, 2),
        "Lost Diff": round(lost_diff, 2),
        "Lost % Change": round(lost_pct, 2) if not np.isnan(lost_pct) else "NA",

        "Current Avg Expiry": round(current_avg_exp, 2),
        "Best Avg Expiry": round(best_avg_exp, 2),
        "Expiry Diff": round(exp_diff, 2),
        "Expiry % Change": round(exp_pct, 2) if not np.isnan(exp_pct) else "NA",

        "Current Avg Profit": round(current_avg_profit, 2),
        "Best Avg Profit": round(best_avg_profit, 2),
        "Profit Diff": round(profit_diff, 2),
        "Profit % Change": round(profit_pct, 2) if not np.isnan(profit_pct) else "NA"
    })

comparison_best_vs_current_df = pd.DataFrame(comparison_rows)

print("\n\n==============================================================")
print("===== AVERAGE COMPARISON: CURRENT vs BEST POLICY (REAL DATA) =====")
print("==============================================================")
print(comparison_best_vs_current_df.to_string(index=False))

# ============================================================
# OVERALL TOTAL COMPARISON
# ============================================================

print("\n\n==============================================================")
print("===== OVERALL TOTAL COMPARISON =====")
print("==============================================================")

current_total_lost = sum(df_real[f"{sku}_lost_sales"].sum() for sku in validated_skus)
best_total_lost = sum(df_best_real[f"{sku}_lost_sales"].sum() for sku in validated_skus)
lost_total_diff = best_total_lost - current_total_lost
lost_total_pct = ((lost_total_diff / current_total_lost) * 100) if current_total_lost != 0 else np.nan

current_total_exp = sum(df_real[f"{sku}_expired"].sum() for sku in validated_skus)
best_total_exp = sum(df_best_real[f"{sku}_expired"].sum() for sku in validated_skus)
exp_total_diff = best_total_exp - current_total_exp
exp_total_pct = ((exp_total_diff / current_total_exp) * 100) if current_total_exp != 0 else np.nan

current_total_profit = sum(df_real[f"{sku}_profit"].sum() for sku in validated_skus)
best_total_profit = sum(df_best_real[f"{sku}_profit"].sum() for sku in validated_skus)
profit_total_diff = best_total_profit - current_total_profit
profit_total_pct = ((profit_total_diff / current_total_profit) * 100) if current_total_profit != 0 else np.nan

overall_summary_df = pd.DataFrame([
    {
        "Metric": "Lost Sales",
        "Current Policy Total": round(current_total_lost, 2),
        "Best Policy Total": round(best_total_lost, 2),
        "Difference": round(lost_total_diff, 2),
        "% Change": round(lost_total_pct, 2) if not np.isnan(lost_total_pct) else "NA"
    },
    {
        "Metric": "Expiry",
        "Current Policy Total": round(current_total_exp, 2),
        "Best Policy Total": round(best_total_exp, 2),
        "Difference": round(exp_total_diff, 2),
        "% Change": round(exp_total_pct, 2) if not np.isnan(exp_total_pct) else "NA"
    },
    {
        "Metric": "Profit",
        "Current Policy Total": round(current_total_profit, 2),
        "Best Policy Total": round(best_total_profit, 2),
        "Difference": round(profit_total_diff, 2),
        "% Change": round(profit_total_pct, 2) if not np.isnan(profit_total_pct) else "NA"
    }
])

print(overall_summary_df.to_string(index=False))

# ============================================================
# FINAL COMPARISON: SIMULATED CURRENT POLICY vs FINAL BEST POLICY
# (USING SAVED RESULTS FROM BOTH BLOCKS)
# ONLY FULLY VALIDATED VEGETABLES
# ============================================================

print("\n\n==============================================================")
print("===== FINAL COMPARISON: CURRENT SIMULATED POLICY vs BEST POLICY =====")
print("==============================================================")

final_compare_rows = []

for sku in validated_skus:

    current_profit = selected_policy_results[sku]["Sim Profit"]
    current_lost = selected_policy_results[sku]["Sim Lost Sales"]
    current_exp = selected_policy_results[sku]["Sim Expiry"]

    best_profit = best_policy_results_final[sku]["Avg Profit"]
    best_lost = best_policy_results_final[sku]["Avg Lost Sales"]
    best_exp = best_policy_results_final[sku]["Avg Expiry"]

    profit_diff = best_profit - current_profit
    lost_diff = best_lost - current_lost
    exp_diff = best_exp - current_exp

    profit_pct = ((profit_diff / current_profit) * 100) if current_profit != 0 else np.nan
    lost_pct = ((lost_diff / current_lost) * 100) if current_lost != 0 else np.nan
    exp_pct = ((exp_diff / current_exp) * 100) if current_exp != 0 else np.nan

    row_ci = validation_15_df[validation_15_df["Vegetable"] == sku].iloc[0]

    profit_ci = row_ci["Profit CI (99%)"]
    lost_ci = row_ci["Lost Sales CI (99%)"]
    exp_ci = row_ci["Expiry CI (99%)"]

    final_compare_rows.append({
        "Vegetable": sku,

        "Current Sim Profit": round(current_profit, 2),
        "Best Sim Profit": round(best_profit, 2),
        "Profit Diff": round(profit_diff, 2),
        "Profit % Change": round(profit_pct, 2) if not np.isnan(profit_pct) else "NA",
        "Profit CI (99%)": profit_ci,

        "Current Sim Lost": round(current_lost, 2),
        "Best Sim Lost": round(best_lost, 2),
        "Lost Diff": round(lost_diff, 2),
        "Lost % Change": round(lost_pct, 2) if not np.isnan(lost_pct) else "NA",
        "Lost Sales CI (99%)": lost_ci,

        "Current Sim Expiry": round(current_exp, 2),
        "Best Sim Expiry": round(best_exp, 2),
        "Expiry Diff": round(exp_diff, 2),
        "Expiry % Change": round(exp_pct, 2) if not np.isnan(exp_pct) else "NA",
        "Expiry CI (99%)": exp_ci
    })

final_compare_df = pd.DataFrame(final_compare_rows)

print(final_compare_df.to_string(index=False))

# ============================================================
# OVERALL TOTAL COMPARISON: CURRENT SIMULATED POLICY vs BEST POLICY
# ONLY FULLY VALIDATED VEGETABLES
# ============================================================

print("\n\n==============================================================")
print("===== OVERALL TOTAL COMPARISON: CURRENT SIMULATED POLICY vs BEST POLICY =====")
print("==============================================================")

current_total_profit = sum(selected_policy_results[sku]["Sim Profit"] for sku in validated_skus)
current_total_lost = sum(selected_policy_results[sku]["Sim Lost Sales"] for sku in validated_skus)
current_total_exp = sum(selected_policy_results[sku]["Sim Expiry"] for sku in validated_skus)

best_total_profit = sum(best_policy_results_final[sku]["Avg Profit"] for sku in validated_skus)
best_total_lost = sum(best_policy_results_final[sku]["Avg Lost Sales"] for sku in validated_skus)
best_total_exp = sum(best_policy_results_final[sku]["Avg Expiry"] for sku in validated_skus)

profit_total_diff = best_total_profit - current_total_profit
lost_total_diff = best_total_lost - current_total_lost
exp_total_diff = best_total_exp - current_total_exp

profit_total_pct = ((profit_total_diff / current_total_profit) * 100) if current_total_profit != 0 else np.nan
lost_total_pct = ((lost_total_diff / current_total_lost) * 100) if current_total_lost != 0 else np.nan
exp_total_pct = ((exp_total_diff / current_total_exp) * 100) if current_total_exp != 0 else np.nan

overall_final_compare_df = pd.DataFrame([
    {
        "Metric": "Profit",
        "Current Sim Total": round(current_total_profit, 2),
        "Best Sim Total": round(best_total_profit, 2),
        "Difference": round(profit_total_diff, 2),
        "% Change": round(profit_total_pct, 2) if not np.isnan(profit_total_pct) else "NA"
    },
    {
        "Metric": "Lost Sales",
        "Current Sim Total": round(current_total_lost, 2),
        "Best Sim Total": round(best_total_lost, 2),
        "Difference": round(lost_total_diff, 2),
        "% Change": round(lost_total_pct, 2) if not np.isnan(lost_total_pct) else "NA"
    },
    {
        "Metric": "Expiry",
        "Current Sim Total": round(current_total_exp, 2),
        "Best Sim Total": round(best_total_exp, 2),
        "Difference": round(exp_total_diff, 2),
        "% Change": round(exp_total_pct, 2) if not np.isnan(exp_total_pct) else "NA"
    }
])

print(overall_final_compare_df.to_string(index=False))

Saving converted_inventory_dataset_inal.xlsx to converted_inventory_dataset_inal.xlsx

===== DISTRIBUTION FITTING =====

Vegetable: Potato
Best Distribution: norm
Mean (μ): 43.7
Variance (σ²): 14.609999999999998
KS Statistic: 0.1727
KS p-value: 0.8792
Fit Status: GOOD FIT

Vegetable: Ivy_gourd
Best Distribution: gamma
Shape (k): 246.9154985943888
Scale (θ): 0.048295922521212634
KS Statistic: 0.2545
KS p-value: 0.4618
Fit Status: GOOD FIT

Vegetable: Cucumber
Best Distribution: triang
Shape (c): 0.9999999503918546
KS Statistic: 0.3
KS p-value: 0.2705
Fit Status: GOOD FIT

Vegetable: Pumpkin
Best Distribution: gamma
Shape (k): 83.48482286252445
Scale (θ): 0.3937862217926291
KS Statistic: 0.1875
KS p-value: 0.8117
Fit Status: GOOD FIT

Vegetable: Bitter_gourd
Best Distribution: gamma
Shape (k): 874304.6438166453
Scale (θ): 0.0043521870144973065
KS Statistic: 0.1456
KS p-value: 0.9638
Fit Status: GOOD FIT

Vegetable: Brinjal
Best Distribution: gamma
Shape (k): 7.513869930322129
Scale (θ): 

/usr/local/lib/python3.12/dist-packages/scipy/stats/_continuous_distns.py:796: RuntimeWarning: invalid value encountered in sqrt
  sk = 2*(b-a)*np.sqrt(a + b + 1) / (a + b + 2) / np.sqrt(a*b)



Vegetable: Plantain
Best Distribution: norm
Mean (μ): 28.4
Variance (σ²): 4.039999999999999
KS Statistic: 0.143
KS p-value: 0.9689
Fit Status: GOOD FIT

Vegetable: Beetroot
Best Distribution: gamma
Shape (k): 3.910483926992285
Scale (θ): 1.690311184447484
KS Statistic: 0.1931
KS p-value: 0.7841
Fit Status: GOOD FIT

Vegetable: Cabbage
Best Distribution: gamma
Shape (k): 320.2117115445992
Scale (θ): 0.2371194682217025
KS Statistic: 0.1282
KS p-value: 0.9891
Fit Status: GOOD FIT

Vegetable: Curry_leaves
Best Distribution: beta
Alpha: 0.6605220189176524
Beta: 1.0752138580086128
KS Statistic: 0.1697
KS p-value: 0.8912
Fit Status: GOOD FIT

Vegetable: Coriander
Best Distribution: norm
Mean (μ): 30.0
Variance (σ²): 7.6
KS Statistic: 0.2416
KS p-value: 0.5271
Fit Status: GOOD FIT

Vegetable: Pudina
Best Distribution: uniform
Lower bound (a): 10.0
Range (b): 3.0
KS Statistic: 0.2333
KS p-value: 0.5708
Fit Status: GOOD FIT

Vegetable: Tomato
Best Distribution: norm
Mean (μ): 33.5
Variance (σ²)